# Vincimap - lancement google colab

Notebook pour lancer `main.py` depuis Google Colab. Avant d'executer, active un runtime GPU dans Colab : `Runtime > Change runtime type > GPU`.

Le notebook separe les trois actions du script : `create_workspace`, `run_colmap`, puis `run_gaussian_training`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Clone du repo

Clone le repo si besoin, puis bascule sur la branche configuree en utilisant `FETCH_HEAD`.

In [ ]:
REPO_URL = 'https://github.com/ZotacD/vincimap.git'
BRANCH = '6-correspondance-image-et-range-distances-calculer-échelle-unité-gsplat-en-untié-mètre'
PROJECT_DIR = '/content/drive/MyDrive/P2iP/GOOGLE_COLLAB/vincimap'

import os
os.environ["PROJECT_DIR"] = PROJECT_DIR

from pathlib import Path

if not Path(PROJECT_DIR).exists():
    !git clone {REPO_URL} {PROJECT_DIR}

%cd {PROJECT_DIR}
!git remote set-url origin {REPO_URL}
!git fetch origin "{BRANCH}"
!git checkout -B "{BRANCH}" FETCH_HEAD
!git reset --hard FETCH_HEAD
!git status --short
!git log -1 --oneline


## Configuration

Cette cellule suppose que le repo a deja ete clone dans `PROJECT_DIR`. Renseigne les chemins des fichiers Drive et du workspace.

In [ ]:
from pathlib import Path
import os

# Fichiers d'entree dans Drive.
VIDEO_PATH = '/content/drive/MyDrive/P2iP/GOOGLE_COLLAB/vincimap/res/bloc_002.MOV'
DISTANCES_PATH = '/content/drive/MyDrive/P2iP/GOOGLE_COLLAB/vincimap/res/bloc_002.txt'

# Workspace de sortie. Le mettre dans Drive conserve les resultats apres la session Colab.
WORKSPACE_PATH = '/content/drive/MyDrive/P2iP/GOOGLE_COLLAB/vincimap/workspaces/bloc_002'

project_path = Path(PROJECT_DIR)
assert project_path.exists(), f'PROJECT_DIR introuvable: {PROJECT_DIR}'
assert Path(VIDEO_PATH).exists(), f'VIDEO_PATH introuvable: {VIDEO_PATH}'
assert Path(DISTANCES_PATH).exists(), f'DISTANCES_PATH introuvable: {DISTANCES_PATH}'

os.chdir(project_path)
print('Projet:', Path.cwd())
print('Workspace:', WORKSPACE_PATH)

## Dependances systeme

In [ ]:
!apt-get update -qq
!DEBIAN_FRONTEND=noninteractive apt-get install -y -qq ffmpeg colmap
!ffmpeg -version | head -n 1
!colmap -h | head -n 5

## Dependances Python

Cette cellule suit exactement l'ordre d'installation du README.

In [ ]:
!python -m pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
!conda create -n py310 python=3.10 -y

In [ ]:
%%bash
cd "$PROJECT_DIR"

conda run -n py310 python -m pip install -q --upgrade pip setuptools wheel

conda run -n py310 python -m pip install -q torch==2.4.1+cu124 torchvision==0.19.1+cu124 torchaudio==2.4.1+cu124 \
  --index-url https://download.pytorch.org/whl/cu124

conda run -n py310 python -m pip install -q ninja numpy jaxtyping rich

conda run -n py310 python -m pip install -q gsplat==1.5.3+pt24cu124 \
  --index-url https://docs.gsplat.studio/whl/pt24cu124

conda run -n py310 python -m pip install --no-build-isolation -r requirements.txt

## Verification runtime

In [ ]:
import torch
print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

%%bash
cd "$PROJECT_DIR"
conda run -n py310 python main.py --help

## 1. Creation du workspace

Cette etape copie la video et le fichier distances, cree les configs, extrait les frames et lie les distances aux images.

In [ ]:
%%bash
cd "$PROJECT_DIR"
conda run -n py310 python main.py \
  --action create_workspace \
  --video_path "{VIDEO_PATH}" \
  --distances_path "{DISTANCES_PATH}" \
  --workspace_path "{WORKSPACE_PATH}" \
  --colmap_path colmap

## 2. Reconstruction COLMAP

Lance `feature_extractor`, `sequential_matcher`, `mapper`, `point_filtering`, `bundle_adjuster`, `image_undistorter`, puis les conversions de modele.

In [ ]:
%%bash
cd "$PROJECT_DIR"
conda run -n py310 python main.py \
  --action run_colmap \
  --workspace_path "{WORKSPACE_PATH}"

## 3. Training Gaussian

In [ ]:
%%bash
cd "$PROJECT_DIR"
conda run -n py310 python main.py \
  --action run_gaussian_training \
  --workspace_path "{WORKSPACE_PATH}"

## Recuperer les sorties

In [ ]:
%%bash
cd "$PROJECT_DIR"
conda run -n py310 find "{WORKSPACE_PATH}" -maxdepth 3 -type f | sort | head -n 80